# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FEZEKIL/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns our validated modeling results into a human-executable **Content Action Playbook**. It acknowledges the measured limits of the model and defines the protocol for human review.

## 1. Ranked actions + reason codes

We use the model to rank content items by decay risk, then attach human-readable reason codes to explain *why* an item is flagged.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
import sys, os

sys.path.append('../../')
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, simple_svg_bar_chart

# 1. Load data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)

# 2. Prep features (as established in W5/W6)
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])

for col in MODEL_NUMERIC_FEATURES: df[col] = df[col].fillna(0)
for col in MODEL_CATEGORICAL_FEATURES: df[col] = df[col].fillna("unknown")

features = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
X = df.copy()
for col in MODEL_CATEGORICAL_FEATURES:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

y = df['is_declining_label']

# 3. Model: Grouped Client Split (Honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X.iloc[train_idx][features], y.iloc[train_idx])

# 4. Generate the Queue
df_playbook = df.copy()
df_playbook['decay_risk_score'] = model.predict_proba(X[features])[:, 1]

# 5. Reason Code Mapping
def get_reason_code(row):
    codes = []
    if row['days_since_last_update'] >= 180: codes.append("STALE_CONTENT")
    if 4 <= row['avg_position'] <= 10: codes.append("STRIKING_DISTANCE")
    if row['impressions_90d'] > df['impressions_90d'].median() and row['ctr'] < df['ctr'].median():
        codes.append("LOW_CTR_HIGH_VIS")
    return "+".join(codes) if codes else "GENERAL_DECAY_RISK"

df_playbook['reason_codes'] = df_playbook.apply(get_reason_code, axis=1)
df_playbook['action'] = "Human Review for Refresh"

queue = df_playbook.sort_values('decay_risk_score', ascending=False)
print(f"Playbook generated: {len(queue)} items ranked.")
queue[['content_id', 'decay_risk_score', 'reason_codes', 'action']].head(10)

## 2. Intended use and limits

**Intended Use:** 
This playbook is a **prioritization tool** for SEO and content teams. It identifies which pages are statistically most likely to be in a period of organic decline based on historical patterns of staleness, visibility, and rank.

**Measured Limits:**
- **Honest Model Skill:** In a client-holdout validation (ML-09), this model achieved a **ROC-AUC of 0.609**. This indicates directional predictive skill but confirms that the model is not a perfect oracle.
- **Memorization Gap:** The gap between random-split (0.772) and grouped-split (0.609) performance shows that a significant portion of the model's apparent skill in standard tests comes from site-specific technical signatures that do not generalize. 
- **Decision-Support Only:** The model flags *risk*, not *guarantee*. It should never be used for autonomous content replacement without human validation.

In [ ]:
# Export metrics for the paper
import json
metrics = {
    "honest_roc_auc": 0.609,
    "random_roc_auc": 0.772,
    "memorization_gap": 0.163,
    "n_clients_train": df_playbook.iloc[train_idx]['client_id'].nunique(),
    "n_clients_test": df_playbook.iloc[test_idx]['client_id'].nunique()
}
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics receipt written to work/outputs/playbook_metrics.json")

## 3. Human review + the no-go list

**Review Protocol (The 3-Step Check):**
1. **Intent Match:** Does the page still answer the primary search intent for its target keyword? If the SERP has shifted (e.g., from informational to commercial), a simple refresh will not work.
2. **Seasonality Audit:** Is the 'decay' actually just a predictable seasonal dip (e.g., 'Best Winter Boots' in July)? 
3. **Technical Integrity:** Is the decline caused by a technical site issue (slow load time, broken layout) rather than content decay?

**The No-Go List (Do NOT Automate):**
- **Legal/Regulatory Pages:** Privacy policies, terms of service, and medical disclosures must be reviewed by subject matter experts, not metrics.
- **Pillar Brand Pages:** High-value landing pages that represent the core brand identity.
- **Zero-Volume/Low-Impression Pages:** The model is highly unstable for pages with < 50 impressions (as seen in ML-08/09 error analysis).

In [ ]:
#Archetype count for the playbook summary
archetypes = queue.head(500)['reason_codes'].value_counts()
print("Archetypes in the Top 500 candidates:")
print(archetypes)

## 4. Monitoring / retrain triggers

The model's recommendations will go stale as Google's algorithms and client portfolios evolve. We monitor for:

1. **Precision@50 Drift:** If monthly manual audits of the top-50 recommendations show that fewer than 50% are valid refresh candidates, the model needs a feature audit.
2. **Feature Distribution Shift:** A significant change in the median `days_since_last_update` across the portfolio (suggesting a site-wide migration or new content strategy).
3. **Base Rate Shift:** If the percentage of pages naturally 'declining' shifts by more than 10 percentage points month-over-month.

In [ ]:
# monitoring logic check
base_rate = df['is_declining_label'].mean()
print(f"Current Portfolio Base Rate (Decline %): {base_rate:.2%}")
print(f"Retrain Trigger: Base rate < {base_rate - 0.1:.2%} or > {base_rate + 0.1:.2%}")

## 5. Exports for the paper

We export the ranked queue and a visual comparison for the final research paper.

In [ ]:
# 1. Export the Queue (Gitignored by design)
queue.to_csv("../outputs/refresh_playbook_queue.csv", index=False)

# 2. Export Figure: Comparison Chart
os.makedirs("../figures", exist_ok=True)
simple_svg_bar_chart(
    title="Model Skill: Honest vs Over-optimistic AUC",
    labels=["Random Split (Memorization)", "Grouped Client Split (Honest)"],
    values=[0.772, 0.609],
    path=pd.io.common.Path("../figures/model_skill_comparison.svg")
)

print("Playbook queue and figures exported to work/outputs/ and work/figures/")

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.